# 05 Create SFINCS Scenarios

## Stage Contract

Requires:
- Event Catalog scenario rows
- accepted dynamic Wflow handoff per staged event (`sfincs_discharge.nc` + QA JSON)
- SFINCS base model domains with native infiltration, roughness, and source points

Produces:
- event-specific SFINCS scenario folders
- native HydroMT-SFINCS rainfall, discharge, and initial-condition files
- cluster launch plan for prepared scenario folders


## Imports and Runtime


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parent
location_name = location_root.name
repo_root = location_root.parents[1]
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from study_location import define_location
from sfincs_runs.scenarios import (
    accepted_dynamic_handoff_event_ids,
    audit_inland_coupled_batch_readiness,
    dynamic_handoff_readiness_table,
    stage_inland_coupled_scenarios,
    stage_inland_coupled_scenario_forcing,
)

definition = define_location(location_root / "config.yaml")
runtime_config = definition.config
config = runtime_config
sfincs_config = runtime_config
wflow_config = {"wflow": runtime_config["wflow"]}
runtime_config["wflow"]["domain_set"]["review_required"] = False
wflow_handoff_manifest = runtime_config.get("wflow", {}).get("handoff", {}).get(
    "manifest",
    "data/wflow/domain_set_handoff.yaml",
)

def resolve_location_path(relative_path):
    return location_root / relative_path

pd.set_option("display.max_colwidth", 140)


## Required Inputs


In [2]:
from wflow_runs.notebook import exists_table
exists_table(location_root, {
    "scenario catalog": "data/event_catalog/catalog/scenario_catalog.csv",
    "probability catalog": "data/event_catalog/catalog/probability_catalog.csv",
    "resilience stress/training catalog": "data/event_catalog/catalog/resilience_stress_training_catalog.csv",
    "event manifest": "data/event_catalog/event_manifest.yaml",
    "wflow handoff manifest": wflow_handoff_manifest,
    "wflow events root": wflow_config["wflow"]["events_root"],
    "sfincs domain manifest": sfincs_config["sfincs_domain_set"]["domain_manifest"],
    "sfincs base model root": sfincs_config["paths"]["base_model_root"],
})


,artifact,path,exists
0,scenario catalog,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.csv,True
1,probability catalog,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/probability_catalog.csv,True
2,resilience stress/training catalog,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/resilience_stress_training_catalog.csv,True
3,event manifest,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/event_manifest.yaml,True
4,wflow handoff manifest,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/domain_set_handoff.yaml,True
5,wflow events root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events,True
6,sfincs domain manifest,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/domains/domain_set.yaml,True
7,sfincs base model root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/base,True


## Scenario Build Plan


In [3]:
scenario_build = sfincs_config["scenario_build"]
pd.Series({
    "design_scenario": scenario_build["design_scenario"],
    "forcing_variable": scenario_build["forcing_variable"],
    "zsini_mode": scenario_build["zsini_mode"],
    "spinup_hours": scenario_build["timing"]["spinup_hours"],
    "drain_down_hours": scenario_build["timing"]["drain_down_hours"],
    "min_run_hours": scenario_build["timing"]["min_run_hours"],
    "max_run_hours": scenario_build["timing"]["max_run_hours"],
})


design_scenario     base
forcing_variable    auto
zsini_mode           dry
spinup_hours          12
drain_down_hours      24
min_run_hours         72
max_run_hours        168
dtype: object

## Wflow Event Forcing


In [4]:
pd.Series({
    "discharge_source": runtime_config["inland_coupling"]["discharge_forcing"]["source"],
    "events_root": wflow_config["wflow"]["events_root"],
    "handoff_manifest": wflow_handoff_manifest,
    "acceptance_artifact": "data/wflow/events/<event_id>/sfincs_discharge.dynamic_handoff.json",
    "sfincs_discharge": "data/wflow/events/<event_id>/sfincs_discharge.nc",
    "zero_rain_control_required": True,
}, name="dynamic_wflow_handoff_contract")


discharge_source                                                                   wflow_dynamic
events_root                                                                    data/wflow/events
handoff_manifest                                              data/wflow/domain_set_handoff.yaml
acceptance_artifact           data/wflow/events/<event_id>/sfincs_discharge.dynamic_handoff.json
sfincs_discharge                                data/wflow/events/<event_id>/sfincs_discharge.nc
zero_rain_control_required                                                                  True
Name: dynamic_wflow_handoff_contract, dtype: object

## Coupled Wflow-SFINCS Cluster Pipeline Prep


In [5]:
# This prepares the cluster worklist and commands for the event-atomic production path:
# Wflow dynamic handoff -> native SFINCS staging -> SFINCS run, per event.
# It does not run Wflow or SFINCS locally.
coupled_pipeline_slurm = "cluster/run_wflow_sfincs_dsai_inland_coupled.slurm"
scenario_root_for_worklists = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
scenario_root_for_worklists.mkdir(parents=True, exist_ok=True)

pipeline_readiness = dynamic_handoff_readiness_table(
    runtime_config,
    location_root,
    catalog_path=resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv"),
    limit=scenario_limit if "scenario_limit" in globals() else None,
)
pipeline_accepted = pipeline_readiness.loc[pipeline_readiness["status"].eq("accepted")].copy()
pipeline_blocked = pipeline_readiness.loc[pipeline_readiness["status"].ne("accepted")].copy()

blocked_worklist_csv = scenario_root_for_worklists / f"{location_name}_blocked_dynamic_handoffs.csv"
accepted_worklist_csv = scenario_root_for_worklists / f"{location_name}_accepted_dynamic_handoffs.csv"
readiness_worklist_csv = scenario_root_for_worklists / f"{location_name}_dynamic_handoff_readiness.csv"
pipeline_readiness.to_csv(readiness_worklist_csv, index=False)
pipeline_blocked.to_csv(blocked_worklist_csv, index=False)
pipeline_accepted.to_csv(accepted_worklist_csv, index=False)

coupled_array_chunks = 32
coupled_max_concurrent_nodes = 8
coupled_debug_event = "<event_id>"

coupled_pipeline_commands = [
    f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only",
    "mkdir -p runs",
    f"sbatch --array=0-0 --export=ALL,LOCATION={location_name},EVENT_IDS={coupled_debug_event},FORCE_RERUN=1,KEEP_STAGE=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm",
    f"sbatch --array=0-{coupled_array_chunks - 1}%{coupled_max_concurrent_nodes} --export=ALL,LOCATION={location_name},WORKLIST=locations/{location_name}/data/sfincs/scenarios/{location_name}_blocked_dynamic_handoffs.csv,FORCE_RERUN=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm",
]
print("# Submit this joint pipeline after syncing inputs to the cluster:")
for command in coupled_pipeline_commands:
    print(command)

display(pd.Series({
    "accepted_dynamic_handoffs": len(pipeline_accepted),
    "events_needing_joint_pipeline": len(pipeline_blocked),
    "readiness_worklist_csv": str(readiness_worklist_csv),
    "blocked_worklist_csv": str(blocked_worklist_csv),
    "accepted_worklist_csv": str(accepted_worklist_csv),
    "coupled_pipeline_slurm": str(repo_root / coupled_pipeline_slurm),
}, name="coupled_wflow_sfincs_cluster_pipeline"))


# Submit this joint pipeline after syncing inputs to the cluster:
SYNC_LOCATION=greensboro cluster/sync_to_dsai.sh --run-inputs-only
mkdir -p runs
sbatch --array=0-0 --export=ALL,LOCATION=greensboro,EVENT_IDS=<event_id>,FORCE_RERUN=1,KEEP_STAGE=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm
sbatch --array=0-31%8 --export=ALL,LOCATION=greensboro,WORKLIST=locations/greensboro/data/sfincs/scenarios/greensboro_blocked_dynamic_handoffs.csv,FORCE_RERUN=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm


accepted_dynamic_handoffs                                                                                                                              1
events_needing_joint_pipeline                                                                                                                        509
readiness_worklist_csv           /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_dynamic_handoff_readiness.csv
blocked_worklist_csv              /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_blocked_dynamic_handoffs.csv
accepted_worklist_csv            /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_accepted_dynamic_handoffs.csv
coupled_pipeline_slurm                                            /home/grahamhults/projects/Flood-RM/cluster/run_wflow_sfincs_dsai_inland_coupled.slurm
Name: coupled_wflow_sfincs_cluster_pipeline, dtype: object

## SFINCS Scenario Folders


In [6]:
stage_scenarios = False  # production uses the joint cluster pipeline above
force_restage = True
scenario_limit = None
require_full_catalog_acceptance = True
scenario_catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
stress_training_catalog_path = resolve_location_path("data/event_catalog/catalog/resilience_stress_training_catalog.csv")
probability_catalog_path = resolve_location_path("data/event_catalog/catalog/probability_catalog.csv")

handoff_readiness = dynamic_handoff_readiness_table(
    runtime_config,
    location_root,
    catalog_path=scenario_catalog_path,
    limit=scenario_limit,
)
accepted_handoffs = handoff_readiness.loc[handoff_readiness["status"].eq("accepted")].copy()
blocked_handoffs = handoff_readiness.loc[handoff_readiness["status"].ne("accepted")].copy()

readiness_dir = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
readiness_dir.mkdir(parents=True, exist_ok=True)
readiness_csv = readiness_dir / f"{location_name}_dynamic_handoff_readiness.csv"
blocked_csv = readiness_dir / f"{location_name}_blocked_dynamic_handoffs.csv"
accepted_csv = readiness_dir / f"{location_name}_accepted_dynamic_handoffs.csv"
handoff_readiness.to_csv(readiness_csv, index=False)
blocked_handoffs.to_csv(blocked_csv, index=False)
accepted_handoffs.to_csv(accepted_csv, index=False)

display(pd.Series({
    "accepted_dynamic_handoffs": len(accepted_handoffs),
    "blocked_events": len(blocked_handoffs),
    "catalog_events_checked": len(handoff_readiness),
    "readiness_csv": str(readiness_csv),
    "blocked_csv": str(blocked_csv),
    "accepted_csv": str(accepted_csv),
}, name="dynamic_handoff_readiness"))
display(accepted_handoffs[["event_id", "sfincs_discharge_forcing", "acceptance"]])
display(blocked_handoffs.head(20))

accepted_event_ids = accepted_dynamic_handoff_event_ids(handoff_readiness)
if stage_scenarios:
    if require_full_catalog_acceptance and not blocked_handoffs.empty:
        raise RuntimeError(
            f"Only {len(accepted_handoffs)} of {len(handoff_readiness)} catalog events have accepted dynamic Wflow handoffs. "
            "Submit the joint Wflow->SFINCS cluster pipeline from the previous cell for the blocked worklist, "
            "then rerun this optional local staging cell only if you need local/prepared scenario folders."
        )
    if not accepted_event_ids:
        raise RuntimeError("No accepted dynamic Wflow handoffs are available. Run 04/b_prepare_wflow_dynamic_handoff.ipynb first.")
    scenario_report = stage_inland_coupled_scenarios(
        runtime_config,
        {"location_root": location_root},
        catalog_path=scenario_catalog_path,
        event_ids=accepted_event_ids,
        force=force_restage,
    )
    forcing_report = stage_inland_coupled_scenario_forcing(
        runtime_config,
        {"location_root": location_root},
        scenario_report=scenario_report,
    )
    display(scenario_report[["event_id", "sfincs_domain_id", "scenario_status", "run_root", "wflow_discharge_forcing"]])
    display(forcing_report[["event_id", "sfincs_domain_id", "status", "n_src", "direct_rainfall_enabled", "netamprfile", "inifile"]])
else:
    scenarios_root = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
    scenario_count = len(pd.read_csv(scenario_catalog_path)) if scenario_catalog_path.exists() else 0
    stress_training_count = len(pd.read_csv(stress_training_catalog_path)) if stress_training_catalog_path.exists() else 0
    scenario_report = pd.Series({
        "stage_scenarios": stage_scenarios,
        "accepted_dynamic_handoffs": len(accepted_event_ids),
        "blocked_events_for_joint_pipeline": len(blocked_handoffs),
        "scenario_catalog_csv": str(scenario_catalog_path),
        "scenario_count": scenario_count,
        "stress_training_catalog_csv": str(stress_training_catalog_path),
        "stress_training_count": stress_training_count,
        "probability_catalog_csv": str(probability_catalog_path),
        "target_root": str(scenarios_root),
        "status": "joint cluster pipeline prepared; local SFINCS staging skipped",
        "next_step": "Submit cluster/run_wflow_sfincs_dsai_inland_coupled.slurm using the printed command above.",
    })
    forcing_report = pd.DataFrame()
    display(scenario_report)


accepted_dynamic_handoffs                                                                                                                          1
blocked_events                                                                                                                                   509
catalog_events_checked                                                                                                                           510
readiness_csv                /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_dynamic_handoff_readiness.csv
blocked_csv                   /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_blocked_dynamic_handoffs.csv
accepted_csv                 /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/greensboro_accepted_dynamic_handoffs.csv
Name: dynamic_handoff_readiness, dtype: object

,event_id,sfincs_discharge_forcing,acceptance
499,design_0398,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0398/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0398/sfincs_discharge.dynamic_handoff.json


,event_id,status,sfincs_discharge_forcing,acceptance,issue
0,design_0007,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0007/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0007/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance for design_0007 is stale: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/de...
1,design_0012,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0012/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0012/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0012: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
2,design_0008,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0008/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0008/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0008: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
3,design_0004,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0004/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0004/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0004: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
4,design_0009,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0009/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0009/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0009: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
5,design_0024,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0024/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0024/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0024: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
6,design_0001,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0001/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0001/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0001: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
7,design_0005,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0005/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0005/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0005: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
8,design_0014,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0014/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0014/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0014: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...
9,design_0019,blocked,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0019/sfincs_discharge.nc,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0019/sfincs_discharge.dynamic_handoff.json,Dynamic Wflow handoff acceptance is missing for design_0019: /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/...


stage_scenarios                                                                                                                                           False
accepted_dynamic_handoffs                                                                                                                                     1
blocked_events_for_joint_pipeline                                                                                                                           509
scenario_catalog_csv                                   /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.csv
scenario_count                                                                                                                                              510
stress_training_catalog_csv          /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/resilience_stress_training_catalog.csv
stress_training_count                   

## Scenario Folder Audit


In [7]:
if accepted_event_ids:
    scenario_audit = audit_inland_coupled_batch_readiness(
        runtime_config,
        {"location_root": location_root},
        catalog_path=scenario_catalog_path,
        event_ids=accepted_event_ids,
    )
    display(scenario_audit["status"].value_counts().rename("readiness_check_count"))
    display(scenario_audit.loc[scenario_audit["status"].ne("passed")].head(30))

    failed_checks = scenario_audit.loc[scenario_audit["status"].ne("passed")]
    if not failed_checks.empty and stage_scenarios:
        raise RuntimeError("SFINCS scenario staging is not end-to-end ready; resolve failed readiness checks before cluster submission.")
else:
    scenario_audit = pd.DataFrame()
    display(pd.Series({"status": "no accepted events to audit yet"}, name="scenario_folder_audit"))

exists_table(location_root, {
    "scenarios root": sfincs_config["paths"]["scenarios_root"],
    "scenario catalog": f"{sfincs_config['paths']['scenarios_root']}/scenario_catalog.csv",
    "scenario forcing report": f"{sfincs_config['paths']['scenarios_root']}/scenario_forcing_report.csv",
    "run stage root": sfincs_config["paths"]["run_root"],
    "run outputs root": sfincs_config["paths"]["storage_root"],
    "stats root": sfincs_config["paths"]["stats_root"],
})


status
passed    13
Name: readiness_check_count, dtype: int64

,event_id,sfincs_domain_id,check,status,path,message


,artifact,path,exists
0,scenarios root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios,True
1,scenario catalog,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/scenario_catalog.csv,True
2,scenario forcing report,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/scenarios/scenario_forcing_report.csv,True
3,run stage root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/run_stage,True
4,run outputs root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/run_outputs,True
5,stats root,/home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/stats,True


## Cluster Launch Plan


In [8]:
# Cluster launch plan + packaging for the inland Wflow->SFINCS scenarios.
# The inland scenarios are nested per SFINCS domain (scenarios/<event_id>/<domain_id>),
# so the array job drives run_events in --scenario-catalog mode off scenario_catalog.csv.
run_settings = sfincs_config["scenario_run"]
scenarios_root = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
run_root = resolve_location_path(sfincs_config["paths"]["run_root"])
storage_root = resolve_location_path(sfincs_config["paths"]["storage_root"])
scenario_catalog_csv = scenarios_root / "scenario_catalog.csv"
slurm_script = "cluster/run_sfincs_dsai_inland_coupled.slurm"

# Cluster array sizing — tune to the partition.
configured_workers = int(run_settings.get("workers", 1) or 1)
workers_per_cpu_node = configured_workers if configured_workers > 1 else 16
array_chunks = 8
max_concurrent_cpu_nodes = 4

if scenario_catalog_csv.exists():
    scenario_catalog = pd.read_csv(scenario_catalog_csv)
    expected_scenario_count = int(len(scenario_catalog))
    debug_event_id = str(scenario_catalog["event_id"].iloc[0]) if expected_scenario_count else "<event_id>"
else:
    expected_scenario_count = None
    debug_event_id = "<event_id>"

package_command = f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only"
commands = [
    package_command,
    "mkdir -p runs runs/debug",
    f"sbatch --array=0-0 --export=ALL,LOCATION={location_name},EVENT_IDS={debug_event_id},WORKERS=1,FORCE_RERUN=1,KEEP_STAGE=1 {slurm_script}",
    f"sbatch --array=0-{array_chunks - 1}%{max_concurrent_cpu_nodes} --export=ALL,LOCATION={location_name},WORKERS={workers_per_cpu_node},FORCE_RERUN=1 {slurm_script}",
]
print("# 1. package + push inputs (code + prepared scenario folders) to the cluster")
print("# 2. one-event debug run, then the full chunked array")
for command in commands:
    print(command)

batch_plan = {
    "location": location_name,
    "scenario_layout": "nested:<event_id>/<sfincs_domain_id>",
    "scenarios_root": str(scenarios_root),
    "scenario_catalog_csv": str(scenario_catalog_csv),
    "scenario_catalog_present": scenario_catalog_csv.exists(),
    "expected_scenario_count": expected_scenario_count,
    "run_root": str(run_root),
    "storage_root": str(storage_root),
    "sfincs_bin": run_settings["sfincs_bin"],
    "sfincs_bin_env": run_settings["sfincs_bin_env"],
    "workers_per_cpu_node": workers_per_cpu_node,
    "array_chunks": array_chunks,
    "max_concurrent_cpu_nodes": max_concurrent_cpu_nodes,
    "slurm_scripts": [str(repo_root / slurm_script)],
    "package_command": package_command,
    "commands": commands,
    "slurm_log_dir": str(repo_root / "runs"),
    "debug_stage_dir": str(repo_root / "runs" / "debug"),
}
run_root.mkdir(parents=True, exist_ok=True)
plan_path = run_root / f"{location_name}_create_scenarios_cluster_plan.json"
plan_path.write_text(json.dumps(batch_plan, indent=2), encoding="utf-8")
display(pd.Series({
    "plan": str(plan_path),
    "slurm_script": str(repo_root / slurm_script),
    "scenario_catalog_present": scenario_catalog_csv.exists(),
    "expected_scenario_count": expected_scenario_count,
    "package_command": package_command,
}, name="cluster_launch_plan"))


# 1. package + push inputs (code + prepared scenario folders) to the cluster
# 2. one-event debug run, then the full chunked array
SYNC_LOCATION=greensboro cluster/sync_to_dsai.sh --run-inputs-only
mkdir -p runs runs/debug
sbatch --array=0-0 --export=ALL,LOCATION=greensboro,EVENT_IDS=design_0398,WORKERS=1,FORCE_RERUN=1,KEEP_STAGE=1 cluster/run_sfincs_dsai_inland_coupled.slurm
sbatch --array=0-7%4 --export=ALL,LOCATION=greensboro,WORKERS=16,FORCE_RERUN=1 cluster/run_sfincs_dsai_inland_coupled.slurm


plan                        /home/grahamhults/projects/Flood-RM/locations/greensboro/data/sfincs/run_stage/greensboro_create_scenarios_cluster_plan.json
slurm_script                                                            /home/grahamhults/projects/Flood-RM/cluster/run_sfincs_dsai_inland_coupled.slurm
scenario_catalog_present                                                                                                                            True
expected_scenario_count                                                                                                                                1
package_command                                                                       SYNC_LOCATION=greensboro cluster/sync_to_dsai.sh --run-inputs-only
Name: cluster_launch_plan, dtype: object